In [1]:
import xarray as xr
import dask.array as da
import numpy as np

In [193]:
ds = xr.Dataset(
    {
        "foo": (("x", "y"), da.empty((10, 10), chunks=(2, 2))),
        "bar": (("x", "y"), da.empty((10, 10), chunks=(2, 2))),
    },
    coords={
        "x": 10*np.arange(10),
        "y": 10*np.arange(10),
    },
)

In [195]:
ds.chunksizes

Frozen({'x': (2, 2, 2, 2, 2), 'y': (2, 2, 2, 2, 2)})

In [192]:
ds.coords

Coordinates:
  * x        (x) int64 80B 0 10 20 30 40 50 60 70 80 90
  * y        (y) int64 80B 0 10 20 30 40 50 60 70 80 90

In [171]:
ds.chunks

Frozen({'x': (2, 2, 2, 2, 2), 'y': (2, 2, 2, 2, 2)})

In [18]:
cs = ds.chunksizes['x']

In [19]:
np.repeat(np.arange(len(cs)), cs)

array([0, 0, 1, 1, 2, 2, 3, 3, 4, 4])

- pixel label: `coord`
- pixel index: `np.arange(len(coord))`
- chunk index: `


In [144]:
def map_pixel_index_to_pixel_index(coords, sel):
    map_ = xr.DataArray(
        np.arange(len(coords)),
        dims=[coords.name],
        coords={coords.name: np.arange(len(coords))},
    )
    return map_.sel({coords.name: sel}).values

In [145]:
def map_pixel_label_to_pixel_index(coord, sel):
    map_ = xr.DataArray(
        data=np.arange(len(coord)),
        coords={coord.name: coord},
    )
    return map_.sel({coord.name: sel}).values

In [146]:
def map_pixel_index_to_pixel_label(coords, sel):
    return coords.isel({coords.name: sel}).values

In [148]:
def map_pixel_index_to_chunk_index(coords, chunksizes, sel):
    map_ = xr.DataArray(
        np.repeat(np.arange(len(chunksizes)), chunksizes),
        dims=[coords.name],
        coords={coords.name: np.arange(len(coords))},
    )
    return map_.isel({coords.name: sel}).values

In [166]:
def map_chunk_index_to_pixel_index(coords, chunksizes, sel):
    map_ = xr.DataArray(
        np.arange(len(coords)),
        dims=[coords.name],
        coords={coords.name: np.repeat(np.arange(len(chunksizes)), chunksizes)},
    )
    return map_.sel({coords.name: sel}).values

In [151]:
map_chunk_index_to_pixel_index(ds.x, ds.chunksizes['x'], slice(3, 4))

array([6, 7, 8, 9])

In [152]:
from abc import ABC, abstractmethod

In [153]:
from abc import ABC, abstractmethod
from enum import Enum
from typing import Any

import xarray as xr
from pydantic import BaseModel
from metaarrays.types import ndarray

class CoordinateSpace(Enum):
    INDEX = 0
    LABEL = 1


class CoordinateLevel(Enum):
    PIXEL = 0
    CHUNK = 1

class CoordinateSpecification(BaseModel):
    level: CoordinateLevel
    space: CoordinateSpace


class MapSpecification(BaseModel):
    from_: CoordinateSpecification
    to: CoordinateSpecification


class PixelToChunkLabelTransform(ABC):
    @abstractmethod
    def transform(
        self, coordinates: xr.DataArray, chunksizes: tuple[int, ...]
    ) -> xr.DataArray:
        """Translate pixel-level labels to chunk-level labels."""


In [163]:

class Mapper(BaseModel):
    coordinate: xr.DataArray
    chunksizes: tuple[int, ...] | None = None
    transform: PixelToChunkLabelTransform | None = None

    class Config:
        arbitrary_types_allowed = True

    def map(self, specification: MapSpecification, value: Any) -> ndarray:
        if self.chunksizes is None:
            if (
                specification.from_.level == CoordinateLevel.CHUNK
                or specification.to.level == CoordinateLevel.PIXEL
            ):
                raise ValueError(
                    "Cannot map at chunk level without chunksizes."
                )

        pixel_index: Any
        match specification.from_:
            case CoordinateSpecification(
                level=CoordinateLevel.PIXEL,
                space=CoordinateSpace.LABEL,
            ):
                pixel_index = map_pixel_label_to_pixel_index(
                    self.coordinate, value
                )
            case CoordinateSpecification(
                level=CoordinateLevel.PIXEL,
                space=CoordinateSpace.INDEX,
            ):
                pixel_index = map_pixel_index_to_pixel_index(
                    self.coordinate, value
                )
            case CoordinateSpecification(
                level=CoordinateLevel.CHUNK,
                space=CoordinateSpace.INDEX,
            ):
                pixel_index = map_chunk_index_to_pixel_index(
                    self.coordinate, self.chunksizes, value
                )
            case CoordinateSpecification(
                level=CoordinateLevel.CHUNK,
                space=CoordinateSpace.LABEL,
            ):
                raise NotImplementedError(
                    "Mapping from chunk label to pixel index is not implemented."
                )
        
        match specification.to:
            case CoordinateSpecification(
                level=CoordinateLevel.PIXEL,
                space=CoordinateSpace.LABEL,
            ):
                return map_pixel_index_to_pixel_label(
                    self.coordinate, pixel_index
                )
            case CoordinateSpecification(
                level=CoordinateLevel.PIXEL,
                space=CoordinateSpace.INDEX,
            ):
                return pixel_index
            case CoordinateSpecification(
                level=CoordinateLevel.CHUNK,
                space=CoordinateSpace.INDEX,
            ):
                return map_pixel_index_to_chunk_index(
                    self.coordinate, self.chunksizes, pixel_index
                )
            case CoordinateSpecification(
                level=CoordinateLevel.CHUNK,
                space=CoordinateSpace.LABEL,
            ):
                raise NotImplementedError(
                    "Mapping from chunk label to pixel index is not implemented."
                )

        

In [164]:
coord = ds.x

In [165]:
Mapper(coordinate=coord, chunksizes=ds.chunks['x']).map(
    MapSpecification(
        from_=CoordinateSpecification(
            level=CoordinateLevel.PIXEL,
            space=CoordinateSpace.INDEX,
        ),
        to=CoordinateSpecification(
            level=CoordinateLevel.CHUNK,
            space=CoordinateSpace.INDEX,
        ),
    ),
    value=slice(3, 10)
)

array([1, 2, 2, 3, 3, 4, 4])